# XGBoost in Python via scikit-learn and 5-fold CV

Author: Daipayan Bera

In [38]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.utils import resample
from functools import reduce
import xgboost as xgb
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score
import time

df = pd.read_csv("PimaIndiansDiabetes2.csv") #I had exported the pimaIndian dataset through R and imported directly here.
ds = df.dropna()

# 2. Fit logistic regression model
# Note: We'll use sklearn which automatically handles logistic regression
X = ds.drop(columns=['diabetes'])
y = (ds['diabetes'] == 'pos').astype(int)  # convert to 0/1

logmodel = LogisticRegression(max_iter=500)
logmodel.fit(X, y)

cfs = np.concatenate(([logmodel.intercept_[0]], logmodel.coef_[0]))  # first intercept, then coefficients
prednames = X.columns.tolist()

# 3. Define the data generator
def data_generator(sz):
    # Sample with replacement for each predictor
    dfdata = pd.DataFrame({
        name: resample(ds[name], n_samples=sz, replace=True, random_state=None).reset_index(drop=True)
        for name in prednames
    })

    # Compute the logit (linear combination of predictors and coefficients)
    pvec = sum(
        cfs[i+1] * dfdata[prednames[i]] for i in range(len(prednames))
    ) + cfs[0]

    # Compute probability using sigmoid function
    probs = 1 / (1 + np.exp(-pvec))

    # Generate outcomes based on probability
    dfdata['outcome'] = (probs > 0.5).astype(int)
    
    return dfdata

In [39]:
def XGboost_training(data):
    X = data.iloc[:, 1:8]  
    y = data["outcome"].values
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    model = xgb.XGBClassifier(
        objective='binary:logistic',  
        max_depth=2,                  
        eta=1,                       
        nthread=2                     
    )
    
    # Cross-validation
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    
    # Fit the model and make predictions
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    # Evaluate accuracy
    accuracy = accuracy_score(y_test, y_pred)
    return accuracy, np.mean(cv_scores)  

score = dict()
execution_times = dict()  #Dictionary to store execution times

for x in [10**i for i in range(2, 8)]:

    start_time = time.time()
    
    generated_data = data_generator(x)
    
    accuracy, mean_cv_score = XGboost_training(generated_data)

    end_time = time.time()
    
    # Calculate the execution time
    execution_time = end_time - start_time
    score[x] = {'accuracy': accuracy, 'mean_cv_score': mean_cv_score}
    execution_times[x] = execution_time  

for size, result in score.items():
    print(f"Data size: {size}, Accuracy: {result['accuracy']:.4f}, Mean CV Score: {result['mean_cv_score']:.4f}, Time: {execution_times[size]:.4f} seconds")

Data size: 100, Accuracy: 0.8500, Mean CV Score: 0.8500, Time: 0.2615 seconds
Data size: 1000, Accuracy: 0.9100, Mean CV Score: 0.9312, Time: 0.3112 seconds
Data size: 10000, Accuracy: 0.9525, Mean CV Score: 0.9484, Time: 0.4470 seconds
Data size: 100000, Accuracy: 0.9619, Mean CV Score: 0.9591, Time: 1.8091 seconds
Data size: 1000000, Accuracy: 0.9620, Mean CV Score: 0.9612, Time: 17.2747 seconds
Data size: 10000000, Accuracy: 0.9608, Mean CV Score: 0.9610, Time: 183.2626 seconds
